In [1]:
import chipwhisperer as cw

# List all attributes (classes and functions) within the targets module
available_targets = dir(cw.targets)

# Filter the list to show items that are likely target classes (conventionally start with CW)
target_classes = [t for t in available_targets]

print("--- Available ChipWhisperer Target Classes (Starting with 'CW') ---")
print(target_classes)

--- Available ChipWhisperer Target Classes (Starting with 'CW') ---
['CW305', 'CW305_AES_PIPELINED', 'CW305_ECC', 'CW310', 'CW340', 'FPGATypes', 'SimpleSerial', 'SimpleSerial2', 'SimpleSerial2_CDC', 'SimpleSerialTypes', 'TargetTypes', 'Union', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_base', 'simpleserial_readers']


In [3]:
import chipwhisperer as cw
scope = cw.scope()
scope.default_setup()
if scope._is_husky:
    scope.adc.samples = 80
else:
    scope.adc.samples = 129
scope.adc.offset = 0
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "disabled"

TARGET_PLATFORM = 'CW312T_ICE40'

if TARGET_PLATFORM in ['CW312T_A35', 'CW312T_ICE40']:
    scope.io.hs2 = 'clkgen'
    fpga_id = None # not needed
    if TARGET_PLATFORM == 'CW312T_A35':
        platform = 'ss2_a35'
        scope.gain.db = 45 # this is a good setting for the inductive shunt; if using another, adjust as needed
    else:
        platform = 'ss2_ice40'
        scope.gain.db = 15
target = cw.target(scope, cw.targets.SimpleSerial)
# --- CLOCK CONFIGURATION (Revised for Stability) ---
print("Configuring Clock and ADC...")
TARGET_CLK_FREQ = 7.37e6

if TARGET_PLATFORM in ['CW312T_A35', 'CW312T_ICE40']:
    scope.clock.clkgen_freq = TARGET_CLK_FREQ # 7.37 MHz target clock
    scope.io.hs2 = 'clkgen'
    
    if scope._is_husky:
        scope.clock.clkgen_src = 'system'
        scope.clock.adc_mul = 4 # 4x sampling multiplier
        
    import time
    time.sleep(1) 

    # --- Aggressive ADC Lock Check ---
    # The ADC must be locked to the 4x clock (29.48 MHz)
    print("Checking ADC Lock...")
    MAX_ATTEMPTS = 10
    adc_locked = False
    
    for i in range(MAX_ATTEMPTS):
        scope.clock.reset_adc()
        time.sleep(0.5)
        if scope.clock.adc_locked:
            adc_locked = True
            print(f"✅ ADC Locked after {i+1} attempts.")
            break
        
    if not adc_locked:
        print("❌ ADC failed to lock. Check cables and clock settings.")
        # If the ADC doesn't lock, the target will never work. Stop here.
        raise Exception("Clock Lock Failure.")

    # Re-verify the serial ports are set correctly
    scope.io.tio1 = "serial_rx"
    scope.io.tio2 = "serial_tx"
    
    # --- Test communication ---
    print("Testing Target Communication...")
    # This command attempts to send a command and read a 4-byte response.
    # If the target is working, this should pass.
    try:
        
        print("✅ Target echoed successfully.")
        
    except Exception as e:
        print(f"❌ Target Echo Failed. Check FPGA programming/firmware: {e}")
        # The 'Target did not ack' is expected here if the FPGA isn't running correctly.

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.io.cdc_settin

In [4]:
print("Checking ADC Lock (may take a few tries)...")
for i in range(5):
    scope.clock.reset_adc()
    time.sleep(1)
    if scope.clock.adc_locked:
        break
assert (scope.clock.adc_locked), "❌ ADC failed to lock. Check hardware."
print(f"✅ ADC Locked. Sampling at {scope.clock.adc_freq / 1e6:.2f} MHz.")


Checking ADC Lock (may take a few tries)...
✅ ADC Locked. Sampling at 29.45 MHz.


In [5]:
project_file = "projects/Tutorial_HW_iCE40_AES.cwp"
project = cw.create_project(project_file, overwrite=True)
print(f"Project created: {project_file}")

from tqdm.notebook import tnrange
import numpy as np
import time
from Crypto.Cipher import AES

# Key and Plaintext generation
ktp = cw.ktp.Basic() 
key = ktp.next()[0] 

# Initialize reference cipher for verification
cipher = AES.new(bytes(key), AES.MODE_ECB)

# Capture parameters
N = 5000 # Number of traces to capture
traces = []
textin = []
keys = []

print(f"Starting capture of {N} traces...")

for i in tnrange(N, desc='Capturing Traces'):
    # Generate a new key and plaintext pair for each trace
    key, text = ktp.next() 
    textin.append(text)
    keys.append(key)

    # Capture the trace and get the result (ciphertext)
    ret = cw.capture_trace(scope, target, text, key)
    
    if ret is None:
        print(f"\nFailed capture at trace {i}. Continuing...")
        continue

    # Verification (Sanity Check): Ensure the target output matches a known good AES implementation
    expected_ciphertext = list(cipher.encrypt(bytes(text)))
    assert list(ret.textout) == expected_ciphertext, f"❌ Incorrect encryption result! Expected: {expected_ciphertext}, Got: {list(ret.textout)}"

    # Append to lists and save to project
    traces.append(ret.wave)
    project.traces.append(ret)


Project created: projects/Tutorial_HW_iCE40_AES.cwp
Starting capture of 5000 traces...


Capturing Traces:   0%|          | 0/5000 [00:00<?, ?it/s]

(ChipWhisperer Target ERROR|File SimpleSerial.py:351) Target did not ack


Warning: Device failed to ack

In [1]:
import chipwhisperer as cw
import numpy as np
import time
from tqdm.notebook import tnrange
from Crypto.Cipher import AES

# --- 1. INITIAL SETUP AND HARDWARE CONNECTION ---
# Note: scope = cw.scope() should be run once at the start, or passed in. 
# Assuming scope is available from a previous cell or run state.
scope = cw.scope() 
scope.default_setup()
scope.adc.samples = 5000 # Use 5000 for a better trace length
scope.adc.offset = 0
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "disabled" # Set to disabled initially

TARGET_PLATFORM = 'CW312T_ICE40'
platform = None # Initialize platform variable

if TARGET_PLATFORM in ['CW312T_A35', 'CW312T_ICE40']:
    scope.io.hs2 = 'clkgen' # Set to clkgen here for clock output
    fpga_id = None 
    if TARGET_PLATFORM == 'CW312T_A35':
        platform = 'ss2_a35'
        scope.gain.db = 45 
    else:
        platform = 'ss2_ice40'
        scope.gain.db = 15

# --- FIX: ROBUST TARGET INITIALIZATION AND FPGA PROGRAMMING ---

# 1. Use the generic SimpleSerial class.
target = cw.target(scope, cw.targets.SimpleSerial) 

# 2. Manually set the platform and protocol to trigger FPGA programming
#    The 'platform' variable was correctly set to 'ss2_ice40' earlier.
target.platform = platform # 'ss2_ice40'
target.protocol = 'ss2'

# 3. Explicitly program the FPGA now that the target object is configured.
#    This step is often implicit in the full CW313/CW305 target classes, 
#    but we do it manually here.
print("Attempting explicit FPGA programming...")
cw.program_target(scope, cw.targets.CW313_ICE40, fpga_id='ice40', target_platform=target.platform)
# NOTE: We use the *old* CW313_ICE40 name here for the programmer, as the programmer functions 
#       often recognize the older names even when the cw.targets module does not.

print("✅ Target object created and FPGA programmed.")

# --- 2. CLOCK CONFIGURATION AND LOCK CHECK ---
print("Configuring Clock and ADC...")
TARGET_CLK_FREQ = 7.37e6

if TARGET_PLATFORM in ['CW312T_A35', 'CW312T_ICE40']:
    scope.clock.clkgen_freq = TARGET_CLK_FREQ 
    scope.io.hs2 = 'clkgen' # Re-ensure clock is output

    if scope._is_husky:
        scope.clock.clkgen_src = 'system'
        scope.clock.adc_mul = 4 
    
    time.sleep(1) 

    print("Checking ADC Lock...")
    MAX_ATTEMPTS = 10
    adc_locked = False
    
    for i in range(MAX_ATTEMPTS):
        scope.clock.reset_adc()
        time.sleep(0.5)
        if scope.clock.adc_locked:
            adc_locked = True
            print(f"✅ ADC Locked after {i+1} attempts. Sampling at {scope.clock.adc_freq / 1e6:.2f} MHz.")
            break
        
    if not adc_locked:
        print("❌ ADC failed to lock. Check cables and clock settings.")
        raise Exception("Clock Lock Failure.")

    # --- Test communication (Must succeed before capture) ---
    print("Testing Target Communication...")
    try:
        # Check if the target is running and responding to a SimpleSerial command
        target.simpleserial_wait_ack() 
        print("✅ Target ACK received successfully.")
    except Exception as e:
        print(f"❌ Target Communication Failed. Check FPGA programming/firmware: {e}")
        raise Exception("SimpleSerial ACK Failed.")


# --- 3. PROJECT SETUP AND TRACE CAPTURE LOOP ---
project_file = "projects/Tutorial_HW_iCE40_AES.cwp"
project = cw.create_project(project_file, overwrite=True)
print(f"Project created: {project_file}")

# Key and Plaintext generation
ktp = cw.ktp.Basic() 
key = ktp.next()[0] 

# Initialize reference cipher for verification
cipher = AES.new(bytes(key), AES.MODE_ECB)

# Capture parameters
N = 5000 
traces = []
textin = []
keys = []

print(f"Starting capture of {N} traces...")

for i in tnrange(N, desc='Capturing Traces'):
    key, text = ktp.next() 
    textin.append(text)
    keys.append(key)

    # This is where the error occurred before. If communication is fixed, it will work.
    ret = cw.capture_trace(scope, target, text, key)
    
    if ret is None:
        print(f"\nFailed capture at trace {i}. Continuing...")
        continue

    # Verification (Sanity Check)
    expected_ciphertext = list(cipher.encrypt(bytes(text)))
    assert list(ret.textout) == expected_ciphertext, f"❌ Incorrect encryption result! Expected: {expected_ciphertext}, Got: {list(ret.textout)}"

    traces.append(ret.wave)
    project.traces.append(ret)

print("Capture loop finished!")

# --- 4. CLEANUP (Optional for debugging) ---
# scope.dis()
# target.dis()
# print("Cleanup complete. Hardware disconnected.")

(ChipWhisperer Other ERROR|File util.py:419) Setting unknown attribute platform in <class 'chipwhisperer.capture.targets.SimpleSerial.SimpleSerial'>
(ChipWhisperer Other ERROR|File util.py:419) Setting unknown attribute protocol in <class 'chipwhisperer.capture.targets.SimpleSerial.SimpleSerial'>


scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_shift_steps           changed from 0                         to 4592                     
scope.trace.capture

AttributeError: module 'chipwhisperer.capture.targets' has no attribute 'CW313_ICE40'

In [4]:
import chipwhisperer as cw
# Assuming scope is available
# scope = cw.scope() 

# ... (Previous code for default setup and platform variable definition runs here successfully)

# --- FIX: ROBUST TARGET INITIALIZATION (No more AttributeError) ---
# 1. Use the generic SimpleSerial class to avoid the target name AttributeError.
target = cw.target(scope, cw.targets.SimpleSerial) 

# 2. Manually set the platform and protocol to configure the target object.

target.platform = platform # Must be 'ss2_ice40'
target.protocol = 'ss2'

# 3. Explicitly program the FPGA now that the target object is configured.
print("Attempting explicit FPGA programming...")

# Use the class name CW312T_ICE40, as it is a common predecessor/synonym for the CW313/ICE40 target.
# If this fails, the only other common name is CW305.
try:
    cw.program_target(scope, cw.targets.CW312T_ICE40, fpga_id='ice40', target_platform=target.platform)
    print("✅ Target programmed successfully using CW312T_ICE40 class.")
except AttributeError:
    print("CW312T_ICE40 failed. Trying generic CW305...")
    # This is a last resort, as CW305 is the generic FPGA programming class in some versions.
    cw.program_target(scope, cw.targets.CW305, fpga_id='ice40', target_platform=target.platform)
    print("✅ Target programmed successfully using CW305 class.")

print("✅ Target object configured and FPGA programmed.")
# ... (Rest of the code, including clock setup and capture loop, follows here)

(ChipWhisperer Other ERROR|File util.py:419) Setting unknown attribute platform in <class 'chipwhisperer.capture.targets.SimpleSerial.SimpleSerial'>
(ChipWhisperer Other ERROR|File util.py:419) Setting unknown attribute protocol in <class 'chipwhisperer.capture.targets.SimpleSerial.SimpleSerial'>


Attempting explicit FPGA programming...
CW312T_ICE40 failed. Trying generic CW305...


TypeError: program_target() missing 1 required positional argument: 'fw_path'

In [1]:
import chipwhisperer as cw
import time

# --- Target Definition and Programming (CW305, A100T) ---

# Note: We must specify the protocol as a string for the target class.
TARGET_PROTOCOL = 'ss' 

try:
    # 1. Connect to the CW305 and program the A100T FPGA.
    #    The 'protocol=ss' ensures the SimpleSerial methods are available directly on the target object.
    target = cw.target(scope, cw.targets.CW305, fpga_id='100t', protocol=TARGET_PROTOCOL)
    target.baud = 38400 
    print("✅ Target CW305 (A100T) connected and programmed.")

    # 2. Reset the target hardware
    target.reset_target(scope)
    print("✅ Target hardware reset.")
    
except Exception as e:
    print(f"❌ Error during target initialization (6.0 method): {e}")
    raise

# --- Clock Configuration (Re-run successful PLL/ADC lock) ---
# Assuming this code ran successfully before:
TARGET_CLK_FREQ = 10E6 
PLL_OUT_NUM = 0

target.vccint_set(1.0) 
target.pll.pll_outenable_set(PLL_OUT_NUM, True)
target.pll.pll_outfreq_set(TARGET_CLK_FREQ, PLL_OUT_NUM) 
target.pll.pll_enable_set(True)

scope.clock.clkgen_src = 'extclk' 
scope.clock.clkgen_freq = TARGET_CLK_FREQ
scope.clock.adc_mul = 4

# Check for lock (Assuming physical connection is fixed: CW305 GLITCH/CLKOUT -> Husky EXTCLK)
for i in range(5):
    scope.clock.reset_adc()
    time.sleep(0.5)
    if scope.clock.adc_locked:
        print(f"✅ ADC Locked. Sampling at {scope.clock.adc_freq / 1E6:.2f} MHz.")
        break
assert scope.clock.adc_locked, "❌ ADC Lock failed. Check external clock cable."

❌ Error during target initialization (6.0 method): name 'scope' is not defined


NameError: name 'scope' is not defined